# Telemetry — generate a run & explore

Same logic as the incidents notebook, for the **telemetry** source. It
reuses the code in `src/sources/telemetry/` and never duplicates it.

- **Section A** reproduces the standard production graphs by calling `src/`.
- **Section B** adds extra ad-hoc analyses, inline.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

from src import config
print("Artifacts dir:", config.TELEMETRY_ARTIFACTS_DIR)

## 1. Generate a fresh run (or pick the latest)

- `GENERATE_NEW_RUN = True` -> run the telemetry pipeline (boxplots + report).
- `GENERATE_NEW_RUN = False` -> only explore the most recent existing run.

In [ ]:
from src.sources.telemetry.runner import run_default

GENERATE_NEW_RUN = False  # True = create a new run; False = explore the latest

if GENERATE_NEW_RUN:
    run_dir = run_default()
else:
    runs = sorted(
        p for p in config.TELEMETRY_ARTIFACTS_DIR.iterdir()
        if p.is_dir() and p.name.isdigit()
    )
    assert runs, "No telemetry run found. Set GENERATE_NEW_RUN = True to create one."
    run_dir = runs[-1]

RUN_ID = run_dir.name
print("Active run:", RUN_ID)

## 2. Load the telemetry data

Telemetry is not transformed, so we load it from the source via the typed
loader (timestamp + numeric columns).

In [ ]:
from src.sources.telemetry.loader import load_telemetry

df = load_telemetry(config.DEFAULT_TELEMETRY_CSV)
df.head()

## 3. Dataset overview

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe().T

## 4. Quality metrics (reuse `src/`)

In [ ]:
from src.common.metrics import compute_quality_metrics

compute_quality_metrics(df)

## A. Standard production graphs (via `src/`)

Calls the exact functions used at each run and displays the saved PNGs
(one boxplot per parameter, distribution per machine). Regenerates the files
inside `run_dir`.

In [ ]:
from src.sources.telemetry import boxplots

for png_path in boxplots.plot_all(df, run_dir):
    display(Image(filename=str(png_path)))

## B. Additional inline analyses

Ad-hoc explorations not part of the standard production set.

### B.1 Correlation between telemetry parameters

In [ ]:
params = [c for c in config.TELEMETRY_PARAM_COLUMNS if c in df.columns]
corr = df[params].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(params)))
ax.set_xticklabels(params, rotation=90)
ax.set_yticks(range(len(params)))
ax.set_yticklabels(params)
for i in range(len(params)):
    for j in range(len(params)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Telemetry parameter correlation")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### B.2 Mean of each parameter per machine

In [ ]:
means = df.groupby(config.MACHINE_COLUMN)[params].mean().round(2)
display(means)

## C. Promoting an analysis to production

When a Section B cell proves useful, move its logic into
`src/sources/telemetry/` and wire it into `runner.py` so it becomes a
standard artifact at every run.